# ✈️ Aviation RAG Assistant
## Fine-tuned Llama-3.2-3B + Pinecone + DOT Official Report

**Doküman:** DOT OIG Flight Delays & Cancellations Report (Oct 2024) — resmi hükümet raporu  
**Model:** Fine-tuned Llama-3.2-3B-Instruct (QLoRA, Dolly-15K)  
**Vector DB:** Pinecone (managed, production-grade)  

```
PDF (DOT raporu)
   ↓ PyPDF → metin çıkar
   ↓ RecursiveCharacterTextSplitter → chunk'la
   ↓ SentenceTransformers → embed et
   ↓ Pinecone → vektör olarak sakla

Kullanıcı sorusu
   ↓ embed et → Pinecone'dan top-3 chunk çek
   ↓ Fine-tuned Llama-3.2-3B + context → cevap üret
```

## 📦 Bölüm 1: Kurulum
Her şeyi tek cell'de kuruyoruz.

In [1]:
!pip install "pinecone-client>=3.0.0" \
            sentence-transformers \
            pypdf \
            langchain-text-splitters \
            transformers>=4.43.0 \
            peft \
            accelerate \
            bitsandbytes \
            unsloth -q

print("✅ Kurulum tamamlandı")

✅ Kurulum tamamlandı


## 📥 Bölüm 2: Import'lar

In [4]:
!pip uninstall pinecone-client -y
!pip install pinecone -q

Found existing installation: pinecone-client 6.0.0
Uninstalling pinecone-client-6.0.0:
  Successfully uninstalled pinecone-client-6.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.9 MB/s eta 0:00:00


In [5]:
import os, time, torch
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from google.colab import userdata

print('✅ Tüm importlar tamamlandı')

✅ Tüm importlar tamamlandı


## 🔑 Bölüm 3: API Keys

Colab sol panel → 🔑 Secrets → şunları ekle:
- `PINECONE_API_KEY` → [pinecone.io](https://pinecone.io) ücretsiz hesap
- `HF_TOKEN` → [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [6]:
from huggingface_hub import login

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
HF_TOKEN         = userdata.get('HF_TOKEN')

login(token=HF_TOKEN)
print('✅ API keys yüklendi')

✅ API keys yüklendi


## 📄 Bölüm 4: PDF Yükle

DOT OIG raporunu Colab'a yükle:  
Sol panel → Files simgesi → Upload (📤) → `dot_flight_delays_2024.pdf`

In [7]:
# PDF'in Colab'a yüklendiğini doğrula
PDF_PATH = '/content/DOT Flight Delays and Cancellations Final Report_10.23.2024.pdf'

if os.path.exists(PDF_PATH):
    size_mb = os.path.getsize(PDF_PATH) / 1024 / 1024
    print(f'✅ PDF bulundu: {PDF_PATH} ({size_mb:.1f} MB)')
else:
    print('❌ PDF bulunamadı!')
    print('Sol panel → Files → Upload ile/content/DOT Flight Delays and Cancellations Final Report_10.23.2024.pdf yükle')

✅ PDF bulundu: /content/DOT Flight Delays and Cancellations Final Report_10.23.2024.pdf (1.1 MB)


## ✂️ Bölüm 5: PDF'ten Metin Çıkar ve Chunk'la

In [8]:
# 1. PDF'ten metin çıkar
reader   = PdfReader(PDF_PATH)
raw_text = ''
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        raw_text += f'\n[Page {i+1}]\n{text}'

print(f'Toplam sayfa:    {len(reader.pages)}')
print(f'Toplam karakter: {len(raw_text):,}')
print(f'\nİlk 500 karakter:\n{raw_text[:500]}')

Toplam sayfa:    28
Toplam karakter: 53,970

İlk 500 karakter:

[Page 1]
Report AV2025003 
October 23, 2024 
The Bureau of Transportation Statistics Verifies the Accuracy 
of Flight Delay and Cancellation Data but Can Do More To 
Assess Its Completeness and Consistency 
BTS
[Page 2]
 
Highlights 
Bureau of Transportation Statistics (BTS ) 
AV2025003 | October 23 , 2024  
www.oig.dot.gov | Contact the Office of Government and Public Affairs at (202)-366-8751 
The Bureau of Transportation Statistics Verifies the Accuracy of Flight 
Delay and Cancellation Data


In [9]:
# 2. Chunk'la
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 500,
    chunk_overlap = 50,
    separators    = ['\n\n', '\n', '. ', ' ']
)
chunks = splitter.split_text(raw_text)
# Çok kısa chunk'ları temizle
chunks = [c for c in chunks if len(c.strip()) >= 50]

print(f'Toplam chunk: {len(chunks)}')
print(f'Ortalama chunk uzunluğu: {sum(len(c) for c in chunks)//len(chunks)} karakter')
print(f'\nÖrnek chunk (5. chunk):\n{chunks[4]}')

Toplam chunk: 120
Ortalama chunk uzunluğu: 454 karakter

Örnek chunk (5. chunk):
• Although BTS requires reporting carriers to certify that the on-time
performance information they submit is correct and complete, the
Agency does not have a method for verifying that carriers are reporting
data for all flights, raising concerns about the data’s reliability.
BTS lacks effective guidance and procedures for verifying reported causes 
data.   
• BTS’ technical directives, which provide guidance to carriers on reporting


## 🔢 Bölüm 6: Embedding Modeli

In [10]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'✅ Embedder hazır — boyut: {embedder.get_sentence_embedding_dimension()}')

# Test
test_emb = embedder.encode('flight delay causes')
print(f'Örnek embedding shape: {test_emb.shape}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder hazır — boyut: 384
Örnek embedding shape: (384,)


## 🌲 Bölüm 7: Pinecone Index

In [11]:
pc         = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = 'aviation-rag'
DIMENSION  = 384

existing = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing:
    print(f'Index {INDEX_NAME} olusturuluyor...')
    pc.create_index(
        name      = INDEX_NAME,
        dimension = DIMENSION,
        metric    = 'cosine',
        spec      = ServerlessSpec(cloud='aws', region='us-east-1')
    )
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print('  Hazirlanıyor...')
        time.sleep(2)
    print('✅ Index olusturuldu')
else:
    print(f'✅ Index zaten mevcut')

index = pc.Index(INDEX_NAME)
print(f'Index stats: {index.describe_index_stats()}')

Index aviation-rag olusturuluyor...
✅ Index olusturuldu
Index stats: {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '150',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 14 Mar 2026 19:32:11 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '44',
                                    'x-pinecone-request-latency-ms': '44',
                                    'x-pinecone-response-duration-ms': '46'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


## 📤 Bölüm 8: Chunk'ları Pinecone'a Yükle

In [12]:
print(f'{len(chunks)} chunk embed ediliyor...')
vectors = []
for i, chunk in enumerate(chunks):
    emb = embedder.encode(chunk).tolist()
    vectors.append({
        'id':     f'dot_chunk_{i:04d}',
        'values': emb,
        'metadata': {
            'content':  chunk[:450],
            'title':    'DOT OIG Flight Delays Report 2024',
            'category': 'Official Government Report',
            'source':   'oig.dot.gov',
        }
    })
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(chunks)} embed edildi...')

# Batch upsert (100'lük)
batch_size = 100
for i in range(0, len(vectors), batch_size):
    index.upsert(vectors=vectors[i:i+batch_size])

time.sleep(2)
stats = index.describe_index_stats()
print(f'\n✅ Pinecone toplam vektör: {stats["total_vector_count"]}')

120 chunk embed ediliyor...
  50/120 embed edildi...
  100/120 embed edildi...

✅ Pinecone toplam vektör: 120


## 🔍 Bölüm 9: Retrieval Fonksiyonu

In [13]:
def retrieve(query: str, top_k: int = 3) -> list:
    vec     = embedder.encode(query).tolist()
    results = index.query(vector=vec, top_k=top_k, include_metadata=True)
    return [
        {
            'content':  m['metadata']['content'],
            'title':    m['metadata']['title'],
            'score':    round(m['score'], 4),
        }
        for m in results['matches']
    ]

# Test
test_qs = [
    'What causes flight delays?',
    'How do weather conditions affect cancellations?',
    'What is the on-time performance rate?',
]
print('Retrieval testi:')
for q in test_qs:
    hits = retrieve(q, top_k=2)
    print(f'\n  Q: {q}')
    for h in hits:
        print(f'    [{h["score"]:.3f}] {h["content"][:80]}...')

Retrieval testi:

  Q: What causes flight delays?
    [0.755] reported causes for flight delays are categorized in five broad categories:   
•...
    [0.634] [Page 22]
Exhibit A. Scope and Methodology 19 
of delayed flights; the annual nu...

  Q: How do weather conditions affect cancellations?
    [0.521] operation of a flight such as thunderstorm, deicing aircraft, tornado,
hurricane...
    [0.439] reported causes for flight delays are categorized in five broad categories:   
•...

  Q: What is the on-time performance rate?
    [0.559] inaccuracies in the on-time performance data it receives from reporting carriers...
    [0.497] Recommendations  
To improve the completeness and consistency of on-time perform...


## 🤖 Bölüm 10: Fine-tuned Llama-3.2-3B Yükle

Bu cell 2-3 dakika sürer.

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

BASE_MODEL    = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER_MODEL = 'Mervecaliskan/llama3.2-3b-dolly-qlora'

print('Tokenizer yukleniyor...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# 4-bit config ayrı tanımlanıyor
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
)

print('Base model yukleniyor (4-bit)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = 'auto',
)

print('LoRA adapter ekleniyor...')
model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL)
model.eval()

print(f'✅ Model hazır — device: {next(model.parameters()).device}')

Tokenizer yukleniyor...
Base model yukleniyor (4-bit)...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

LoRA adapter ekleniyor...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

✅ Model hazır — device: cuda:0


## 🔗 Bölüm 11: RAG Pipeline

In [17]:
def rag_answer(question: str, top_k: int = 3, max_new_tokens: int = 300) -> dict:
    # 1. Retrieve
    docs    = retrieve(question, top_k=top_k)
    context = '\n\n'.join([f"[{d['title']}]\n{d['content']}" for d in docs])

    # 2. Prompt
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        'You are an aviation operations assistant. '
        'Answer questions using only the provided context. '
        'Be concise and factual. Cite specific numbers when available.<|eot_id|>\n'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'Context:\n{context}\n\nQuestion: {question}<|eot_id|>\n'
        '<|start_header_id|>assistant<|end_header_id|>\n'
    )

    # 3. Generate
    inputs = tokenizer(
        prompt, return_tensors='pt',
        truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.3,
            top_p              = 0.9,
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.1,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    answer     = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        'question': question,
        'answer':   answer,
        'sources':  [{'content': d['content'][:100], 'score': d['score']} for d in docs],
    }

print('✅ RAG pipeline hazır')

✅ RAG pipeline hazır


## 🧪 Bölüm 12: Test

In [18]:
test_questions = [
    'What are the main causes of flight delays?',
    'How does weather affect flight cancellations?',
    'What recommendations does the report make to improve on-time performance?',
    'Which airlines have the worst delay rates?',
    'What is the role of FAA in reducing flight delays?',
]

print('Aviation RAG Assistant — Test')
print('=' * 70)

for q in test_questions:
    print(f'\nQ: {q}')
    r = rag_answer(q)
    print(f'A: {r["answer"][:300]}')
    print(f'Top source: {r["sources"][0]["content"][:80]}...')
    print('-' * 70)

Aviation RAG Assistant — Test

Q: What are the main causes of flight delays?
A: The main causes of flight delays are categorized into five broad categories: 
1. Air carrier (Airline)
2. Extreme weather
3. NAS
4. Late-arriving aircraft
5. Security
Top source: reported causes for flight delays are categorized in five broad categories:   
•...
----------------------------------------------------------------------

Q: How does weather affect flight cancellations?
A: Weather can cause flight cancellations if it is extreme.
Top source: operation of a flight such as thunderstorm, deicing aircraft, tornado,
hurricane...
----------------------------------------------------------------------

Q: What recommendations does the report make to improve on-time performance?
A: The report makes two recommendations to improve on-time performance: update and formalize BTS's standard operating procedures for collecting, processing, and disseminating airline on-time performance data and make sure that thes

## 📊 Bölüm 13: RAG vs No-RAG Karşılaştırması

In [19]:
def answer_no_rag(question: str) -> str:
    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n'
        'You are an aviation assistant.<|eot_id|>\n'
        '<|start_header_id|>user<|end_header_id|>\n'
        f'{question}<|eot_id|>\n'
        '<|start_header_id|>assistant<|end_header_id|>\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=150, temperature=0.3,
            do_sample=True, pad_token_id=tokenizer.eos_token_id
        )
    new = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()

q = 'What does the DOT report say about airline on-time performance improvements?'

print('RAG vs No-RAG')
print('=' * 70)
print(f'Soru: {q}\n')
print('WITHOUT RAG (sadece model parametrik bilgisi):')
print(f'  {answer_no_rag(q)[:250]}')
print('\nWITH RAG (DOT raporu context):')
r = rag_answer(q)
print(f'  {r["answer"][:250]}')
print(f'\n→ RAG rapordan spesifik bulgu ve öneri içermeli')

RAG vs No-RAG
Soru: What does the DOT report say about airline on-time performance improvements?

WITHOUT RAG (sadece model parametrik bilgisi):
  The Department of Transportation (DOT) released its annual report on airline on-time performance, which showed that 77.4% of flights were on-time in 2022, up from 74.3% in 2021. The report also noted that the number of on-time flights increased by 1.

WITH RAG (DOT raporu context):
  The DOT report states that improved consistency in BTS data would allow the agency to evaluate carriers' operating and scheduling performance and take actions to benefit airline customers or improve the efficiency of the National Air Traffic Control 

→ RAG rapordan spesifik bulgu ve öneri içermeli
